In [ ]:
from importlib import reload
from IPython.core.interactiveshell import InteractiveShell
%load_ext autoreload
InteractiveShell.ast_node_interactivity = "all"
import logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [29]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath("/cmnfs/proj/ORIGINS/protMSD/maxquant/ScanByScan/swaps")
if module_path not in sys.path:
    sys.path.append(module_path)

# Run SWAPS with example config

In [ ]:
%autoreload 2
!python /cmnfs/proj/ORIGINS/protMSD/maxquant/ScanByScan/sbs_runner_ims.py --config_path=/cmnfs/proj/ORIGINS/protMSD/maxquant/ScanByScan/utils/exp_configs/config_ayla_test.yaml

/cmnfs/home/z.xiao/miniconda3/envs/sbs/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2024-12-19 13:15:18> merge with cfg file /cmnfs/proj/ORIGINS/protMSD/maxquant/ScanByScan/utils/exp_configs/config_ayla_test_ecoli.yaml
2024-12-19 13:15:18> ==================Load data==================
2024-12-19 13:15:18> Reading mzML file
2024-12-19 13:15:25> Saving data to pickle file
2024-12-19 13:15:32> Filtered reference maxquant result by raw file: ['BBM_647_P241_02_07_ssDDA_MIA_001']
2024-12-19 13:15:32> Using multiple GPUs, device is gpu
2024-12-19 13:15:33> maxquant_exp_df size: (95925, 61)
2024-12-19 13:15:33> maxquant_exp_df size after filter by raw file ['BBM_647_P241_02_07_ssDDA_MIA_001']: (19186, 61)
2024-12-19 13:15:33> maxquant_exp_df size after removing matched precursors: (19186, 61)
2024-12-19 13:15:33> maxquant_ref_df size after re

# Check peptide activation

In [5]:
import sparse

pept_act = sparse.load_npz(
    "/cmnfs/proj/ORIGINS/SWAPS_exp/test_thermo/test_ayla_20241219_131745_378483/results/activation/im_rt_pept_act_coo_peptbatch0.npz"
)
# convert to dense
pept_act_mat = pept_act.todense()
pept_act_mat.shape

(2987, 1227)

In [8]:
pept_act_sum_by_scan = pept_act_mat.sum(axis=1)
pept_act_sum_by_pept = pept_act_mat.sum(axis=0)

pept_act_mat has shape (n_ms1scans + 1, n_candidate + 1). The last row is a place holder so it is alway zero. To map the rows back to actual retention time value, you can use the mzML file. Row idx is the same as mzML scan index.

The first column is a place holder candidate since the candidates are identified by 'mz_rank', there is no rank 0 so column 0 always have zero intensity. You can map back the peptide using the dictionary from the result_path.

In [12]:
# Load dictionary
maxquant_ref = pd.read_pickle(
    "/cmnfs/proj/ORIGINS/SWAPS_exp/test_thermo/test_ayla_20241219_131745_378483/maxquant_result_ref.pkl"
)
maxquant_ref.columns
# mz_rank is the column that can be used for mapping candidate

Index(['Sequence', 'Length', 'Modifications', 'Modified sequence',
       'Oxidation (M) Probabilities', 'Oxidation (M) Score Diffs',
       'Oxidation (M)', 'Missed cleavages', 'Proteins', 'Leading proteins',
       'Leading razor protein', 'Type', 'Raw file', 'Experiment', 'MS/MS m/z',
       'Charge', 'm/z', 'Mass', 'Resolution',
       'Uncalibrated - Calibrated m/z [ppm]',
       'Uncalibrated - Calibrated m/z [Da]', 'Mass Error [ppm]',
       'Mass Error [Da]', 'Uncalibrated Mass Error [ppm]',
       'Uncalibrated Mass Error [Da]', 'Max intensity m/z 0', 'Retention time',
       'Retention length', 'Calibrated retention time',
       'Calibrated retention time start', 'Calibrated retention time finish',
       'Retention time calibration', 'Match time difference',
       'Match m/z difference', 'Match q-value', 'Match score',
       'Number of data points', 'Number of scans', 'Number of isotopic peaks',
       'PIF', 'Fraction of total spectrum', 'Base peak fraction', 'PEP',
    

# Add extra candidates

In [13]:
# Load dataframe from JSON file
df_from_json = pd.read_json(
    "/cmnfs/data/proteomics/origin/ayla_proteometools_subset/01709a_GB1-TUM_first_pool_100_01_01-DDA-1h-R1_mz.json"
)
print(df_from_json.head())

            mz   rt_start     rt_end  tolerance tolerance_unit  comment
0  1013.491028  49.734975  50.115053          5            ppm        2
1   675.996444  49.734975  50.115053          5            ppm        3
2   507.249152  49.734975  50.115053          5            ppm        4
3   406.000777  49.734975  50.115053          5            ppm        5
4   930.464966  52.804498  53.536034          5            ppm       12


These extra precursor charge states need to be appended to the MaxQuant Evidence.txt.

Maybe directly copy the identified precursors and modify the following column for different charge state:
- 'Charge'
- 'm/z'
- 'Retention time'
- 'Retention length'
- 'Calibrated retention time'
- 'Calibrated retention time start'
- 'Calibrated retention time finish'
- 'id'  # simply extend this, in principle not used by SWAPS
- 'Intensity'  # set to zero


and leave these columns empty:
- 'MS/MS m/z',
- 'Uncalibrated 
- Calibrated m/z [ppm]',
- 'Uncalibrated - Calibrated m/z [Da]'
- 'Mass Error [ppm]'
- 'Mass Error [Da]'
- 'Uncalibrated Mass Error [ppm]'
- 'Uncalibrated Mass Error [Da]'
- 'Max intensity m/z 0'
- 'Retention time calibration'
- 'Match time difference'
- 'Match m/z difference'
- 'Match q-value'
- 'Match score'
- 'Number of data points'
- 'Number of scans'
- 'Number of isotopic peaks'
- 'PIF'
- 'Fraction of total spectrum'
- 'Base peak fraction'
- 'PEP'
- 'MS/MS Count'
- 'MS/MS Scan Number'
- 'Score'
- 'Delta score'
- 'Combinatorics'
- 'MS/MS IDs'
- 'Best MS/MS'
- 'AIF MS/MS IDs'


In [14]:
mq_evidence = pd.read_csv(
    "/cmnfs/data/proteomics/origin/ayla_proteometools_subset/TUM_first_pool_100_01_01_DDA-1h-R1-tryptic/evidence.txt",
    sep="\t",
)
mq_evidence.columns

Index(['Sequence', 'Length', 'Modifications', 'Modified sequence',
       'Oxidation (M) Probabilities', 'Oxidation (M) Score Diffs',
       'Oxidation (M)', 'Missed cleavages', 'Proteins', 'Leading proteins',
       'Leading razor protein', 'Type', 'Raw file', 'Experiment', 'MS/MS m/z',
       'Charge', 'm/z', 'Mass', 'Resolution',
       'Uncalibrated - Calibrated m/z [ppm]',
       'Uncalibrated - Calibrated m/z [Da]', 'Mass Error [ppm]',
       'Mass Error [Da]', 'Uncalibrated Mass Error [ppm]',
       'Uncalibrated Mass Error [Da]', 'Max intensity m/z 0', 'Retention time',
       'Retention length', 'Calibrated retention time',
       'Calibrated retention time start', 'Calibrated retention time finish',
       'Retention time calibration', 'Match time difference',
       'Match m/z difference', 'Match q-value', 'Match score',
       'Number of data points', 'Number of scans', 'Number of isotopic peaks',
       'PIF', 'Fraction of total spectrum', 'Base peak fraction', 'PEP',
    